# Runner — Anatomy-Aware Pose Estimation (STL Training)
Notebook **minimale**: tutto il codice vero sta nei moduli `.py` sul repo GitHub.

**Prima di lanciare:**
1. Settings → **Internet: On**
2. **Add Input** → COCO 2017 Keypoints + OCHuman + dataset `pose-baseline-checkpoint`

In [6]:
# === 1. Codice dal repo GitHub ===
import os, sys

REPO_URL = "https://github.com/flaviomassaroni/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks.git"
REPO_DIR = "/kaggle/working/repo"

!rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}

mod_dir = None
for root, dirs, files in os.walk(REPO_DIR):
    if ".git" in root: continue
    if "config.py" in files:
        mod_dir = root
        break
if mod_dir:
    sys.path.insert(0, mod_dir)
    print("Moduli:", sorted([f for f in os.listdir(mod_dir) if f.endswith(".py")]))
else:
    print("ERRORE: config.py non trovato!")

fatal: could not create leading directories of '/kaggle/working/repo': Permission denied
ERRORE: config.py non trovato!


In [7]:
import stl, inspect
print("BONE_SCALE" in dir(stl))                    # True solo col file nuovo
print("_logcosh" in dir(stl))                       # idem
src = inspect.getsource(stl.bone_ratio_loss)
print("logcosh" in src and "log-cosh" in src)       # True col nuovo

True
True
True


In [8]:
# === 2. Import + seed + check ===
from config import *
import utils, data, network, train
import evaluation as ev
from stl import SkeletalTopologyLoss
import cv2
cv2.setNumThreads(0)  # Forza OpenCV a non spawnare thread propri

set_seed(SEED)
print("Device:", device)
print("Ambiente:", ENV_NAME)
print("COCO val ann:", COCO_VAL_ANN)

# Listing dataset montati: solo su Kaggle. In locale /kaggle/input non esiste.
if os.path.isdir("/kaggle/input"):
    print("\n/kaggle/input contiene:")
    for d in os.listdir("/kaggle/input"):
        print("  -", d)
else:
    print("\n(locale: /kaggle/input non presente, uso path da config)")

Device: cuda
Ambiente: LOCAL
COCO val ann: /home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/anatomy-aware-pose/datasets/coco2017/annotations/person_keypoints_val2017.json

(locale: /kaggle/input non presente, uso path da config)


In [9]:
# === 3. Dati COCO ===
from torch.utils.data import DataLoader

train_samples = data.build_samples(COCO_TRAIN_ANN, min_keypoints=8)
val_samples   = data.build_samples(COCO_VAL_ANN)
print(f"train: {len(train_samples)} | val: {len(val_samples)}")

train_dataset = data.COCOKeypointsDataset(train_samples, COCO_TRAIN_IMG, INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)
val_dataset   = data.COCOKeypointsDataset(val_samples,   COCO_VAL_IMG,   INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

loading annotations into memory...
Done (t=3.70s)
creating index...
index created!
loading annotations into memory...
Done (t=0.08s)
creating index...
index created!
train: 116021 | val: 6352


In [10]:
import config, importlib
importlib.reload(config)
BEST_PTH = config.BEST_PTH
print("BEST_PTH:", BEST_PTH, "| esiste:", os.path.exists(BEST_PTH))

BEST_PTH: /home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/anatomy-aware-pose/models/best.pth | esiste: True


In [ ]:
if False:
    import torch
    from stl import soft_argmax
    model.eval()
    ratios_all = []
    valids_all = []
    with torch.no_grad():
        for bi,(imgs,hms,w) in enumerate(train_loader):
            if bi>=8: break
            imgs,w = imgs.to(device), w.to(device)
            c = soft_argmax(model(imgs), beta=10.0)
            def blen(c,a,b): return torch.sqrt(((c[:,a]-c[:,b])**2).sum(-1)+1e-6)
            r = blen(c,7,9)/(blen(c,5,7)+1e-6)
            v = (w.squeeze(-1)[:,5]>0)&(w.squeeze(-1)[:,7]>0)&(w.squeeze(-1)[:,9]>0)
            ratios_all.append(r[v].cpu())
    r = torch.cat(ratios_all)
    qs = torch.quantile(r, torch.tensor([0.01,0.05,0.25,0.5,0.75,0.95,0.99]))
    print("quantili forearm/upper_arm (solo validi):")
    for p,q in zip([1,5,25,50,75,95,99], qs): print(f"  p{p:02d}: {q:.3f}")

In [ ]:
# === 3c. Calibrazione lambda gradient-norm (Strada B) ===
# Misura ||grad L_t / d(final_layer)|| per ogni termine NON pesato
# e per la heatmap loss. Lambda calibrati come:
#   lambda_t = STL_TARGET_FRAC * g_hm / (g_t + eps)
# Cosi ogni termine imprime ai pesi una frazione TARGET_FRAC della
# spinta della heatmap loss (influenza equalizzata, non valore equalizzato).
# Ref: Chen et al. GradNorm 2018. Vedi stl.py::calibrate_lambdas per i dettagli.
import torch
from stl import SkeletalTopologyLoss, calibrate_lambdas

model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(BEST_PTH, map_location=device))

diag_criterion = SkeletalTopologyLoss(
    heatmap_criterion=train.WeightedMSELoss(),
    lambda_bone=1.0, lambda_angle=1.0, lambda_order=1.0, lambda_collapse=1.0,
    beta=STL_BETA,
)

# Se COCO train non e'ancora scaricato, campiona dal val_loader (1 riga)
try:
    calib_loader = train_loader
except NameError:
    calib_loader = val_loader
    print("ATTENZIONE: uso val_loader per calibrazione (COCO train non disponibile)")

lam_suggest = calibrate_lambdas(
    diag_criterion, model, calib_loader, device,
    target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True,
)
# calibrate_lambdas stampa norme grad + lambda calibrati.
# lam_suggest dict viene letto dalla cella 4b.


In [ ]:
# === 4a. Training BASELINE (gia' completato — NON eseguire) ===
if False:
    import torch
    NUM_EPOCHS = 30
    model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=True).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[15, 25], gamma=0.1)
    criterion = train.WeightedMSELoss()
    history = train.fit(model, train_loader, val_loader, optimizer, scheduler, criterion, device, NUM_EPOCHS, CHECKPOINT_DIR, resume=True)

In [ ]:
# === 4b-SANITY. Sanity check 1 epoca PRIMA del run completo ===
# Esegue UNA epoca di fine-tuning STL da best.pth e verifica due guardie
# (sez. 12.1 del README) contro la baseline:
#   G1) AP deve reggere entro ~1 punto:  AP >= 0.488   (baseline 0.498)
#   G2) AVR deve gia' scendere:          AVR < 0.306   (baseline 0.306)
# Se una guardia fallisce, NON lanciare la cella 9: qualcosa nel harness
# (lambda, LR, o gate STL/AVR) sta ancora rompendo il training.
# Questa epoca parte da best.pth e salva in una dir separata: NON sporca
# il run definitivo, che ricomincia comunque da best.pth.
import torch, os, time
from tqdm.auto import tqdm
import evaluation as ev
from stl import SkeletalTopologyLoss, calibrate_lambdas

# --- Baseline di riferimento (dal README, sez. 10/11.9) ---
BASE_AP  = 0.498
BASE_AVR = 0.306
AP_TOL   = 0.010          # "entro ~1 punto"
AP_FLOOR = BASE_AP - AP_TOL   # 0.488

# --- BEST_PTH portabile (stessa logica della cella 9) ---
try:
    _best = BEST_PTH
except NameError:
    _best = None
if not _best or not os.path.exists(_best):
    _best = "/kaggle/input/datasets/messinaalberto/pose-baseline-checkpoint/best.pth"
assert os.path.exists(_best), f"best.pth non trovato: {_best}"
print("Baseline checkpoint:", _best)

model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(_best, map_location=device))
print("Pesi baseline caricati")

LR_STL = STL_FINE_TUNE_LR

# --- Calibrazione lambda (identica alla cella 9, sul modello baseline) ---
criterion = SkeletalTopologyLoss(
    heatmap_criterion=train.WeightedMSELoss(),
    lambda_bone=1.0, lambda_angle=1.0, lambda_order=1.0, lambda_collapse=1.0,
    beta=STL_BETA,
)
print("Calibrazione lambda (gradient-norm, su baseline)...")
lam = calibrate_lambdas(
    criterion, model, train_loader, device,
    target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True,
)
print(f"  lambda: bone={criterion.lambda_bone:.5f} angle={criterion.lambda_angle:.5f} "
      f"order={criterion.lambda_order:.5f} collapse={criterion.lambda_collapse:.5f}")

# Azzero i gradienti sporchi lasciati dalla calibrazione PRIMA dell'optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=LR_STL)
optimizer.zero_grad(set_to_none=True)

SANITY_DIR = "/kaggle/working/checkpoints_sanity"
os.makedirs(SANITY_DIR, exist_ok=True)

# --- UNA epoca di training ---
t0 = time.time()
model.train()
sums = {k: 0.0 for k in ['heatmap', 'bone', 'angle', 'order', 'collapse', 'total']}
for imgs, hms, w in tqdm(train_loader, desc="sanity 1 epoca", leave=False):
    imgs, hms, w = imgs.to(device), hms.to(device), w.to(device)
    optimizer.zero_grad()
    out = model(imgs)
    loss, terms = criterion(out, hms, w)
    loss.backward()
    optimizer.step()
    for k in sums: sums[k] += terms[k] * imgs.size(0)
n = len(train_loader.dataset)
for k in sums: sums[k] /= n

torch.save(model.state_dict(), f"{SANITY_DIR}/sanity_e01.pth")

# --- Valutazione AP + AVR su COCO val ---
model.eval()
coco_eval, avr = ev.evaluate_on_coco_val(
    model, val_samples, device,
    results_path=f"{SANITY_DIR}/coco_val_pred_sanity.json")
ap  = float(coco_eval.stats[0])
ar  = float(coco_eval.stats[5])
avr_rate = avr['AVR_pose_rate']

print(f"\nE01 (sanity) tot:{sums['total']:.4f} hm:{sums['heatmap']:.4f} "
      f"bone:{sums['bone']:.4f} ang:{sums['angle']:.4f} "
      f"ord:{sums['order']:.4f} col:{sums['collapse']:.4f} "
      f"| AP:{ap:.4f} AR:{ar:.4f} AVR:{avr_rate:.4f} | {time.time()-t0:.0f}s")

# --- VERDETTO ---
g1 = ap >= AP_FLOOR
g2 = avr_rate < BASE_AVR
print("\n" + "=" * 56)
print("VERDETTO SANITY CHECK")
print("=" * 56)
print(f"  G1  AP regge      : AP {ap:.4f} >= {AP_FLOOR:.4f}  -> "
      f"{'PASS' if g1 else 'FAIL'}   (baseline {BASE_AP:.4f})")
print(f"  G2  AVR scende    : AVR {avr_rate:.4f} < {BASE_AVR:.4f} -> "
      f"{'PASS' if g2 else 'FAIL'}   (baseline {BASE_AVR:.4f})")
print("-" * 56)
if g1 and g2:
    print("  ESITO: PASS — puoi lanciare la cella 9 (run completo).")
    print(f"         AP {BASE_AP:.4f}->{ap:.4f} ({ap-BASE_AP:+.4f}), "
          f"AVR {BASE_AVR:.4f}->{avr_rate:.4f} ({avr_rate-BASE_AVR:+.4f})")
else:
    print("  ESITO: FAIL — NON lanciare il run completo. Diagnosi:")
    if not g1:
        print("    - AP crollato in 1 epoca: LR/lambda troppo aggressivi, la STL")
        print("      sta distruggendo le heatmap (catastrophic forgetting).")
    if not g2:
        print("    - AVR non scende (o sale): sintomo del mismatch gate STL/AVR")
        print("      (STL su target_weight GT, AVR su score>=MIN_CONF predetto).")
        print("      La loss migliora le coord soft-argmax sui kp annotati mentre")
        print("      le coord MISURATE (argmax) peggiorano. Fermati e rivedi il gate.")
print("=" * 56)


In [ ]:
# === 4b. Fine-tuning con STL (calibrazione lambda integrata + selezione su AP COCO val) ===
import torch, os, time
from tqdm.auto import tqdm
import evaluation as ev
from stl import SkeletalTopologyLoss, calibrate_lambdas

# --- BEST_PTH portabile: rispetta quello del config se esiste, altrimenti
#     cade sul path Kaggle del dataset condiviso. Cosi la stessa cella gira
#     sia in locale (config locale definisce BEST_PTH) sia su Kaggle. ---
try:
    _best = BEST_PTH
except NameError:
    _best = None
if not _best or not os.path.exists(_best):
    _best = "/kaggle/input/datasets/messinaalberto/pose-baseline-checkpoint/best.pth"
assert os.path.exists(_best), f"best.pth non trovato: {_best}"
print("Baseline checkpoint:", _best)

model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(_best, map_location=device))
print('Pesi baseline caricati')

# --- Iperparametri fine-tuning (tutti da config.py) ---
NUM_EPOCHS_STL = STL_NUM_EPOCHS
LR_STL = STL_FINE_TUNE_LR   # 3e-5: evita catastrophic forgetting (era 1e-4)

# --- Calibrazione lambda INTEGRATA (non dipende piu' dalla cella 3c) ---
# Misura ||grad L_t / d(final_layer)|| per ogni termine non pesato e per la
# heatmap loss, poi lambda_t = STL_TARGET_FRAC * g_hm / (g_t + eps), con spread
# clamp (max 20x) per stabilita'. Vedi stl.py::calibrate_lambdas.
# La eseguo SUL MODELLO BASELINE appena caricato, prima del fine-tuning, cosi
# i lambda riflettono lo stato da cui partiamo. Se lam_suggest esiste gia'
# (cella 3c eseguita a mano), lo rispetto e salto la ricalibrazione.
criterion = SkeletalTopologyLoss(
    heatmap_criterion=train.WeightedMSELoss(),
    lambda_bone=1.0, lambda_angle=1.0, lambda_order=1.0, lambda_collapse=1.0,
    beta=STL_BETA,
)

try:
    lam = lam_suggest  # gia' calibrato dalla cella 3c
    criterion.lambda_bone     = lam['bone']
    criterion.lambda_angle    = lam['angle']
    criterion.lambda_order    = lam['order']
    criterion.lambda_collapse = lam['collapse']
    print("Uso lambda gia' calibrati dalla cella 3c")
except NameError:
    print('Calibrazione lambda in corso (gradient-norm, su baseline)...')
    lam = calibrate_lambdas(
        criterion, model, train_loader, device,
        target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True,
    )
    # calibrate_lambdas aggiorna criterion in-place e ritorna i lambda.

print(f"  lambda finali: bone={criterion.lambda_bone:.5f} "
      f"angle={criterion.lambda_angle:.5f} order={criterion.lambda_order:.5f} "
      f"collapse={criterion.lambda_collapse:.5f}")

# IMPORTANTE: calibrate_lambdas lascia gradienti sporchi sul modello.
# Azzero prima di creare l'optimizer e iniziare il training.
optimizer = torch.optim.Adam(model.parameters(), lr=LR_STL)
optimizer.zero_grad(set_to_none=True)

# milestone a 70% delle epoche: si scala automaticamente se cambi NUM_EPOCHS_STL
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[round(0.7 * NUM_EPOCHS_STL)], gamma=0.1)

STL_CKPT_DIR = '/kaggle/working/checkpoints_stl'
os.makedirs(STL_CKPT_DIR, exist_ok=True)

# --- Criterio di selezione: AP COCO val, NON val_loss ---
# val_loss include la STL: minimizzarla premia un modello con STL bassa anche
# se l'AP e' peggiorata. L'obiettivo del progetto e' abbassare AVR SENZA
# distruggere AP, quindi seleziono sull'AP (stats[0] di pycocotools).
# Salvo comunque ogni epoca, cosi la scelta finale resta ispezionabile a mano.
best_ap = -1.0
history = []

for epoch in range(1, NUM_EPOCHS_STL + 1):
    t0 = time.time()
    model.train()
    sums = {k: 0.0 for k in ['heatmap', 'bone', 'angle', 'order', 'collapse', 'total']}
    for imgs, hms, w in tqdm(train_loader, desc=f'train {epoch}', leave=False):
        imgs, hms, w = imgs.to(device), hms.to(device), w.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss, terms = criterion(out, hms, w)
        loss.backward()
        optimizer.step()
        for k in sums: sums[k] += terms[k] * imgs.size(0)
    n = len(train_loader.dataset)
    for k in sums: sums[k] /= n

    # Salvo SEMPRE il checkpoint di questa epoca (scelta finale ispezionabile)
    torch.save(model.state_dict(), f'{STL_CKPT_DIR}/epoch_{epoch:02d}.pth')

    # Valuto AP+AVR su COCO val per la selezione
    model.eval()
    coco_eval, avr = ev.evaluate_on_coco_val(
        model, val_samples, device,
        results_path=f'{STL_CKPT_DIR}/coco_val_pred_e{epoch:02d}.json')
    ap = float(coco_eval.stats[0])      # AP @ IoU=0.50:0.95
    ar = float(coco_eval.stats[5])      # AR @ IoU=0.50:0.95
    avr_rate = avr['AVR_pose_rate']
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    print(f"E{epoch:02d} tot:{sums['total']:.4f} hm:{sums['heatmap']:.4f} "
          f"bone:{sums['bone']:.4f} ang:{sums['angle']:.4f} "
          f"ord:{sums['order']:.4f} col:{sums['collapse']:.4f} "
          f"| AP:{ap:.4f} AR:{ar:.4f} AVR:{avr_rate:.4f} "
          f"lr:{lr:.0e} {time.time()-t0:.0f}s")
    history.append({'epoch': epoch, 'AP': ap, 'AR': ar, 'AVR': avr_rate,
                    **{f'L_{k}': sums[k] for k in sums}})

    if ap > best_ap:
        best_ap = ap
        torch.save(model.state_dict(), f'{STL_CKPT_DIR}/best_stl.pth')
        print(f'  -> nuovo best STL (AP {ap:.4f}, AVR {avr_rate:.4f})')

print('\nFine-tuning completato.')
print('Storia per-epoca (scegli a mano il trade-off AP/AVR che preferisci):')
for h in history:
    print(f"  E{h['epoch']:02d}  AP {h['AP']:.4f}  AR {h['AR']:.4f}  AVR {h['AVR']:.4f}")

# NB: best_stl.pth = miglior AP. Se un'epoca con AP di poco inferiore ha un AVR
# molto migliore, e' un trade-off legittimo per il paper: i checkpoint epoch_NN.pth
# sono tutti su disco per ricaricare quello che preferisci.


In [11]:
# === SEVERITY WEIGHTING (lineare) — tutto in cella, no modifiche a stl.py ===
#
# Diagnosi: la hinge sui casi correggibili (ratio 1.7-3.0) ha gradiente diluito
# dalla media di batch (~4.3x sui violatori, ma diluito). Il severity LINEARE
# pesa la hinge per la gravita': peso = 1 + alpha*excess. Rinforza la spinta sui
# casi correggibili SENZA accanirsi sugli estremi (foreshortening floor) — al
# contrario del focal (scartato: gradiente ~0 sui medi, massimo sugli estremi).
# Dataset-independent: pesa per quanto una REGOLA FISSA e' violata, non impara
# statistiche dai dati -> tesi del paper preservata.
#
# Confronto target: A_pure_E8 deterministico (COCO AVR 0.3016, OC 0.4745).
import math, os, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import evaluation as ev
# riuso gli helper di stl.py (fedeli all'originale)
from stl import (_bone_length, _kp_valid, _masked_mean, _logcosh, BONE_SCALE,
                 soft_argmax, joint_angle_loss, geometric_ordering_loss,
                 collapse_loss, calibrate_lambdas)
from anthropometric_constraints import BONE_RATIOS, SYMMETRY_PAIRS

# ---- PARAMETRO severity ----
SEV_ALPHA = 2.0           # peso = 1 + alpha*excess; 0 = hinge originale
BOOST_BONE = 5.0          # stesso boost di A
LR = 1e-5                 # stesso LR di A (vincente)
N_EPOCHS = 8

# ---- bone_ratio_loss col severity lineare sul sotto-termine simmetria ----
def bone_ratio_loss_sev(coords, valid_mask, alpha=SEV_ALPHA):
    losses = []
    # (a) inter-segmentali: IDENTICO all'originale (log-cosh, non toccato)
    for name, rule in BONE_RATIOS.items():
        log_nom = math.log(rule['nominal'])
        na0, nb0 = rule['numerator']; da0, db0 = rule['denominator']
        for off in [0, 1]:
            na, nb, da, db = na0+off, nb0+off, da0+off, db0+off
            mask = _kp_valid(valid_mask, na, nb, da, db)
            ratio = _bone_length(coords, na, nb) / (_bone_length(coords, da, db) + 1e-6)
            z = (torch.log(ratio + 1e-6) - log_nom) / BONE_SCALE
            losses.append(_masked_mean(_logcosh(z), mask))
    # (b) simmetria sx/dx: hinge * (1 + alpha*excess)  <-- SEVERITY LINEARE
    LOG_SYM = math.log(1.5)
    for (la, lb), (ra, rb), _ in SYMMETRY_PAIRS:
        mask = _kp_valid(valid_mask, la, lb, ra, rb)
        ratio = _bone_length(coords, la, lb) / (_bone_length(coords, ra, rb) + 1e-6)
        excess = F.relu(torch.log(ratio + 1e-6).abs() - LOG_SYM)
        base = torch.clamp(excess**2, max=4.0)
        penalty = base * (1.0 + alpha * excess)     # severity lineare
        losses.append(_masked_mean(penalty, mask))
    return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=coords.device)

# ---- loss combinata che usa bone_ratio_loss_sev ----
class STL_Severity(nn.Module):
    def __init__(self, heatmap_criterion, lambda_bone, lambda_angle,
                 lambda_order, lambda_collapse, beta, alpha=SEV_ALPHA):
        super().__init__()
        self.hm = heatmap_criterion
        self.lb, self.la, self.lo, self.lc = lambda_bone, lambda_angle, lambda_order, lambda_collapse
        self.beta, self.alpha = beta, alpha
        # attributi compatibili con calibrate_lambdas
        self.heatmap_criterion = heatmap_criterion
    @property
    def lambda_bone(self): return self.lb
    @lambda_bone.setter
    def lambda_bone(self, v): self.lb = v
    @property
    def lambda_angle(self): return self.la
    @lambda_angle.setter
    def lambda_angle(self, v): self.la = v
    @property
    def lambda_order(self): return self.lo
    @lambda_order.setter
    def lambda_order(self, v): self.lo = v
    @property
    def lambda_collapse(self): return self.lc
    @lambda_collapse.setter
    def lambda_collapse(self, v): self.lc = v
    def forward(self, pred, target, tw):
        L_hm = self.hm(pred, target, tw)
        coords = soft_argmax(pred, beta=self.beta)
        vm = tw.squeeze(-1)
        L_bone = bone_ratio_loss_sev(coords, vm, self.alpha)
        L_ang  = joint_angle_loss(coords, vm)
        L_ord  = geometric_ordering_loss(coords, vm)
        L_col  = collapse_loss(coords, vm)
        L = L_hm + self.lb*L_bone + self.la*L_ang + self.lo*L_ord + self.lc*L_col
        terms = {'heatmap':L_hm.item(),'bone':L_bone.item(),'angle':L_ang.item(),
                 'order':L_ord.item(),'collapse':L_col.item(),'total':L.item()}
        return L, terms

# NOTA: calibrate_lambdas chiama bone_ratio_loss (originale) internamente per la
# norma del gradiente. Va bene: calibra sul bone GREZZO, il severity amplifica
# poi in training. Per coerenza pesiamo comunque col boost dopo la calibrazione.

# ---- DETERMINISMO ----
from config import set_seed, SEED
set_seed(SEED)
g = torch.Generator(); g.manual_seed(SEED)
train_loader_det = DataLoader(train_dataset, batch_size=32, shuffle=True,
                              num_workers=0, pin_memory=True, generator=g)

_best = BEST_PTH
model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(_best, map_location=device))
print(f"Baseline: {_best} | Ambiente: {ENV_NAME} | SEV_ALPHA={SEV_ALPHA}")

criterion = STL_Severity(train.WeightedMSELoss(), 1.0, 1.0, 1.0, 1.0,
                         beta=STL_BETA, alpha=SEV_ALPHA)
print("Calibrazione lambda...")
calibrate_lambdas(criterion, model, train_loader_det, device,
                  target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True)
lam_bone_calib = criterion.lambda_bone
criterion.lambda_bone = lam_bone_calib * BOOST_BONE
print(f"lambda_bone: {lam_bone_calib:.5f} -> {criterion.lambda_bone:.5f} ({BOOST_BONE}x)\n")

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
optimizer.zero_grad(set_to_none=True)
OUT_DIR = os.path.abspath(os.path.join(CHECKPOINT_DIR, "..", f"sev_lin_a{SEV_ALPHA}"))
os.makedirs(OUT_DIR, exist_ok=True)

model.eval()
_, avr0 = ev.evaluate_on_coco_val(model, val_samples, device, results_path=f"{OUT_DIR}/pred_e00.json")
pc0 = avr0['per_category']
print(f"{'ep':>3} {'AP':>7} {'AVR':>7} {'bone':>7} {'angle':>7} {'colla':>7}  {'L_bone':>8}  time  note")
print("-"*78)
print(f"{'0':>3} {'-':>7} {avr0['AVR_pose_rate']:>7.4f} {pc0['bone_ratio']:>7.4f} "
      f"{pc0['joint_angle']:>7.4f} {pc0['collapse']:>7.4f}   (baseline E0)")

BASE_COCO_AVR = 0.3065      # baseline deterministica
A_PURE_COCO   = 0.3016      # target da battere (A_pure_E8 deterministico)
best = (1e9, None, None)
for epoch in range(1, N_EPOCHS+1):
    t0 = time.time(); model.train()
    sums = {k:0.0 for k in ['heatmap','bone','total']}
    for imgs,hms,w in tqdm(train_loader_det, desc=f"sev e{epoch}", leave=False):
        imgs,hms,w = imgs.to(device),hms.to(device),w.to(device)
        optimizer.zero_grad()
        loss,terms = criterion(model(imgs), hms, w)
        loss.backward(); optimizer.step()
        for k in sums: sums[k]+=terms[k]*imgs.size(0)
    n=len(train_loader_det.dataset)
    for k in sums: sums[k]/=n
    torch.save(model.state_dict(), f"{OUT_DIR}/epoch_{epoch:02d}.pth")
    model.eval()
    ce,avr = ev.evaluate_on_coco_val(model, val_samples, device, results_path=f"{OUT_DIR}/pred_e{epoch:02d}.json")
    ap=float(ce.stats[0]); pc=avr['per_category']; ar=avr['AVR_pose_rate']
    note=""
    if ar<best[0] and ap>=0.486: best=(ar,epoch,ap); note=" *best"
    if ar<A_PURE_COCO: note+=" <BATTE A_pure"
    print(f"{epoch:>3} {ap:>7.4f} {ar:>7.4f} {pc['bone_ratio']:>7.4f} {pc['joint_angle']:>7.4f} "
          f"{pc['collapse']:>7.4f}  {sums['bone']:>8.4f}  {time.time()-t0:.0f}s{note}")

print("\n"+"="*70)
print(f"VERDETTO severity lineare (alpha={SEV_ALPHA})")
print("="*70)
print(f"  miglior AVR COCO: {best[0]:.4f} @E{best[1]} (AP {best[2]:.4f})")
print(f"  baseline det: {BASE_COCO_AVR} | A_pure_E8 det: {A_PURE_COCO}")
if best[0] < A_PURE_COCO:
    print(f"  BATTE A_pure di {best[0]-A_PURE_COCO:+.4f}. Valuta epoch_{best[1]:02d} su OCHuman.")
elif best[0] < BASE_COCO_AVR:
    print(f"  Meglio della baseline ma non di A_pure. Severity non aggiunge su COCO.")
else:
    print(f"  Non migliora. Severity lineare non aiuta su COCO.")
print("="*70)

Baseline: /home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/anatomy-aware-pose/models/best.pth | Ambiente: LOCAL | SEV_ALPHA=2.0
Calibrazione lambda...
Calibrazione lambda (gradient-norm, rho=0.1, 4 batch, ref=final_layer):
  norme grad grezze: heatmap=2.103e-04  bone=1.856e-01  angle=2.409e-02  order=2.501e+00  collapse=4.987e-04
  lambda calibrati : bone=0.00011  angle=0.00087  order=0.00010  collapse=0.00194
lambda_bone: 0.00011 -> 0.00057 (5.0x)



inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.09s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.09s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=11.18s).
Accumulating evaluation results...
DONE (t=0.03s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.498
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.777
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.532
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.483
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.523
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.539
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.800
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.576
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | m

sev e1:   0%|          | 0/3626 [00:00<?, ?it/s]

inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

...

inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.08s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.09s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.79s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.496
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.777
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.532
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.483
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.521
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.537
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.799
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.576
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | ma

sev e8:   0%|          | 0/3626 [00:00<?, ?it/s]

inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=20.50s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.16s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.80s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.497
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.776
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.532
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.483
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.521
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.538
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.799
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.577
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | m

In [ ]:
# === 5. Valutazione: confronto baseline vs STL ===
import torch

model_base = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model_base.load_state_dict(torch.load(BEST_PTH, map_location=device))
print("=== BASELINE ===")
coco_base, avr_coco_base = ev.evaluate_on_coco_val(model_base, val_samples, device)
oc_base, avr_oc_base = ev.evaluate_on_ochuman(model_base, device)

model_stl = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model_stl.load_state_dict(torch.load(f'{STL_CKPT_DIR}/best_stl.pth', map_location=device))
print("\n=== STL ===")
coco_stl, avr_coco_stl = ev.evaluate_on_coco_val(model_stl, val_samples, device,
    results_path='/kaggle/working/coco_val_pred_stl.json')
oc_stl, avr_oc_stl = ev.evaluate_on_ochuman(model_stl, device,
    results_path='/kaggle/working/ochuman_pred_stl.json')

LABELS = ["AP","AP.50","AP.75","AP_M","AP_L","AR","AR.50","AR.75","AR_M","AR_L"]
header = f"{'':15}{'COCO base':>10}{'COCO STL':>10}{'':3}{'OC base':>10}{'OC STL':>10}"
print(f"\n{header}")
print("-" * 58)
for i, lab in enumerate(LABELS):
    cb = coco_base.stats[i]; cs = coco_stl.stats[i]
    ob = oc_base.stats[i]; os_ = oc_stl.stats[i]
    print(f"{lab:15}{cb:10.4f}{cs:10.4f}{'':3}{ob:10.4f}{os_:10.4f}")
ab = avr_coco_base['AVR_pose_rate']; as_ = avr_coco_stl['AVR_pose_rate']
ob = avr_oc_base['AVR_pose_rate']; os_ = avr_oc_stl['AVR_pose_rate']
print(f"\n{'AVR rate':15}{ab:10.4f}{as_:10.4f}{'':3}{ob:10.4f}{os_:10.4f}")
ab = avr_coco_base['AVR_mean_violations']; as_ = avr_coco_stl['AVR_mean_violations']
ob = avr_oc_base['AVR_mean_violations']; os_ = avr_oc_stl['AVR_mean_violations']
print(f"{'AVR mean':15}{ab:10.4f}{as_:10.4f}{'':3}{ob:10.4f}{os_:10.4f}")

In [12]:
# === EVAL SEVERITY E7/E8 vs baseline vs A_pure (deterministico, COCO+OCHuman) ===
import os, json
import numpy as np
import torch
from torch.utils.data import DataLoader
from evaluation import run_coco_eval, evaluate_avr, build_samples
from data import COCOEvalDataset
from utils import decode_heatmaps, heatmap_to_original

set_seed(SEED)

def run_inference_det(model, samples, img_dir, device, batch_size=32):
    ds = COCOEvalDataset(samples, img_dir, INPUT_SIZE)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    model.eval(); coco_results, coords_all, scores_all = [], [], []
    with torch.no_grad():
        for imgs, image_ids, bboxes in loader:
            imgs = imgs.to(device); heatmaps = model(imgs)
            coords_hm, scores = decode_heatmaps(heatmaps)
            bn = bboxes.numpy(); idn = image_ids.numpy()
            for i in range(imgs.shape[0]):
                ci = heatmap_to_original(coords_hm[i], bn[i], INPUT_SIZE, HEATMAP_SIZE)
                ks = scores[i]; flat = []
                for k in range(NUM_KEYPOINTS):
                    flat += [float(ci[k,0]), float(ci[k,1]), float(ks[k])]
                coco_results.append({'image_id':int(idn[i]),'category_id':1,'keypoints':flat,'score':float(ks.mean())})
                coords_all.append(ci); scores_all.append(ks)
    return coco_results, coords_all, scores_all

_oc = build_samples(OCHUMAN_VAL_ANN)
def ev_coco(m, j):
    r,c,s = run_inference_det(m, val_samples, COCO_VAL_IMG, device)
    return run_coco_eval(r, COCO_VAL_ANN, j), evaluate_avr(c, s)
def ev_oc(m, j):
    r,c,s = run_inference_det(m, _oc, OCHUMAN_IMG, device)
    return run_coco_eval(r, OCHUMAN_VAL_ANN, j), evaluate_avr(c, s)

CKPT = os.path.abspath(os.path.join(CHECKPOINT_DIR, ".."))
SEV_DIR = os.path.join(CKPT, "sev_lin_a2.0")
MODELS = {
    'baseline':    BEST_PTH,
    'A_pure_E8':   os.path.join(CKPT, "configA_ext8", "epoch_08.pth"),
    'sev_E7':      os.path.join(SEV_DIR, "epoch_07.pth"),
    'sev_E8':      os.path.join(SEV_DIR, "epoch_08.pth"),
}
for n,p in MODELS.items():
    print(f"{n:12} {'OK' if os.path.exists(p) else 'MANCANTE'}  {p}")

OUT = os.path.join(CKPT, "eval_severity"); os.makedirs(OUT, exist_ok=True)
def load(p):
    m = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
    m.load_state_dict(torch.load(p, map_location=device)); m.eval(); return m

rows = {}
for name, path in MODELS.items():
    if not os.path.exists(path): print(f"salto {name}"); continue
    print(f"\n===== {name} =====")
    m = load(path)
    print("-- COCO --"); ce_c, avr_c = ev_coco(m, f"{OUT}/{name}_coco.json")
    print("-- OCHuman --"); ce_o, avr_o = ev_oc(m, f"{OUT}/{name}_oc.json")
    rows[name] = {'coco_ap':float(ce_c.stats[0]),'coco_avr':avr_c['AVR_pose_rate'],
                  'oc_ap':float(ce_o.stats[0]),'oc_avr':avr_o['AVR_pose_rate'],
                  'oc_pc':avr_o['per_category'],'oc_mean':avr_o['AVR_mean_violations']}
    del m; torch.cuda.empty_cache()

print("\n"+"="*72)
print("CONFRONTO (deterministico)")
print("="*72)
print(f"{'model':>12} | {'COCO AP':>8} {'COCO AVR':>9} | {'OC AP':>7} {'OC AVR':>8} {'OC mean':>8}")
print("-"*72)
for n,r in rows.items():
    print(f"{n:>12} | {r['coco_ap']:>8.4f} {r['coco_avr']:>9.4f} | {r['oc_ap']:>7.4f} {r['oc_avr']:>8.4f} {r['oc_mean']:>8.4f}")
print("-"*72)
b = rows.get('baseline'); a = rows.get('A_pure_E8')
if b:
    print("\nDELTA vs baseline (negativo AVR = meglio):")
    print(f"{'model':>12} | {'dCOCO_AVR':>10} | {'dOC_AVR':>9}")
    for n,r in rows.items():
        if n=='baseline': continue
        print(f"{n:>12} | {r['coco_avr']-b['coco_avr']:>+10.4f} | {r['oc_avr']-b['oc_avr']:>+9.4f}")
if a:
    print(f"\nvs A_pure_E8 (OC AVR {a['oc_avr']:.4f}):")
    for n in ['sev_E7','sev_E8']:
        if n in rows:
            d = rows[n]['oc_avr']-a['oc_avr']
            mark = " BATTE A_pure su OCHuman" if d<0 else " peggio di A_pure su OCHuman"
            print(f"  {n}: OC AVR {rows[n]['oc_avr']:.4f} ({d:+.4f}){mark}")
print("\nper-categoria OCHuman:")
print(f"{'model':>12} | {'bone':>8} {'angle':>8} {'collapse':>9}")
for n,r in rows.items():
    pc=r['oc_pc']; print(f"{n:>12} | {pc['bone_ratio']:>8.4f} {pc['joint_angle']:>8.4f} {pc['collapse']:>9.4f}")
json.dump(rows, open(f"{OUT}/results_severity.json","w"), indent=2)
print(f"\nSalvato: {OUT}/results_severity.json")
print("="*72)

loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
baseline     OK  /home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/anatomy-aware-pose/models/best.pth
A_pure_E8    OK  /home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/anatomy-aware-pose/configA_ext8/epoch_08.pth
sev_E7       OK  /home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/anatomy-aware-pose/sev_lin_a2.0/epoch_07.pth
sev_E8       OK  /home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/anatomy-aware-pose/sev_lin_a2.0/epoch_08.pth

===== baseline =====
-- COCO --
loading annotations into memory...
Done (t=0.08s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.09s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoint